In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score, average_precision_score
from sklearn.impute import SimpleImputer

In [2]:
df = pd.read_csv('data/training.csv')

In [3]:
df.head()

,RefId,IsBadBuy,PurchDate,Auction,VehYear,VehicleAge,Make,Model,Trim,SubModel,...,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,PRIMEUNIT,AUCGUART,BYRNO,VNZIP1,VNST,VehBCost,IsOnlineSale,WarrantyCost
0,1,0,12/7/2009,ADESA,2006,3,MAZDA,MAZDA3,i,4D SEDAN I,...,11597.0,12409.0,NaN,NaN,21973,33619,FL,7100.0,0,1113
1,2,0,12/7/2009,ADESA,2004,5,DODGE,1500 RAM PICKUP 2WD,ST,QUAD CAB 4.7L SLT,...,11374.0,12791.0,NaN,NaN,19638,33619,FL,7600.0,0,1053
2,3,0,12/7/2009,ADESA,2005,4,DODGE,STRATUS V6,SXT,4D SEDAN SXT FFV,...,7146.0,8702.0,NaN,NaN,19638,33619,FL,4900.0,0,1389
3,4,0,12/7/2009,ADESA,2004,5,DODGE,NEON,SXT,4D SEDAN,...,4375.0,5518.0,NaN,NaN,19638,33619,FL,4100.0,0,630
4,5,0,12/7/2009,ADESA,2005,4,FORD,FOCUS,ZX3,2D COUPE ZX3,...,6739.0,7911.0,NaN,NaN,19638,33619,FL,4000.0,0,1020


In [4]:
df.shape

(72983, 34)

In [5]:
df.columns

Index(['RefId', 'IsBadBuy', 'PurchDate', 'Auction', 'VehYear', 'VehicleAge',
       'Make', 'Model', 'Trim', 'SubModel', 'Color', 'Transmission',
       'WheelTypeID', 'WheelType', 'VehOdo', 'Nationality', 'Size',
       'TopThreeAmericanName', 'MMRAcquisitionAuctionAveragePrice',
       'MMRAcquisitionAuctionCleanPrice', 'MMRAcquisitionRetailAveragePrice',
       'MMRAcquisitonRetailCleanPrice', 'MMRCurrentAuctionAveragePrice',
       'MMRCurrentAuctionCleanPrice', 'MMRCurrentRetailAveragePrice',
       'MMRCurrentRetailCleanPrice', 'PRIMEUNIT', 'AUCGUART', 'BYRNO',
       'VNZIP1', 'VNST', 'VehBCost', 'IsOnlineSale', 'WarrantyCost'],
      dtype='object')

In [6]:
df['PurchDate'] = pd.to_datetime(df['PurchDate'])

df = df.sort_values('PurchDate').reset_index(drop=True)

total_days = (df['PurchDate'].max() - df['PurchDate'].min()).days
train_end = df['PurchDate'].min() + pd.Timedelta(days=total_days // 3)
valid_end = train_end + pd.Timedelta(days=total_days // 3)

print(f"Общий период: {df['PurchDate'].min()} - {df['PurchDate'].max()}")
print(f"Train до: {train_end}")
print(f"Validation: {train_end} - {valid_end}")
print(f"Test после: {valid_end}")

train_df = df[df['PurchDate'] <= train_end].copy()
valid_df = df[(df['PurchDate'] > train_end) & (df['PurchDate'] <= valid_end)].copy()
test_df = df[df['PurchDate'] > valid_end].copy()

print(f"\nРазмеры выборок:")
print(f"Train: {len(train_df)} объектов")
print(f"Validation: {len(valid_df)} объектов")
print(f"Test: {len(test_df)} объектов")

print(f"\nПроверка дат:")
print(f"Train max: {train_df['PurchDate'].max()}")
print(f"Validation min: {valid_df['PurchDate'].min()}")
print(f"Validation max: {valid_df['PurchDate'].max()}")
print(f"Test min: {test_df['PurchDate'].min()}")

Общий период: 2009-01-05 00:00:00 - 2010-12-30 00:00:00
Train до: 2009-09-03 00:00:00
Validation: 2009-09-03 00:00:00 - 2010-05-02 00:00:00
Test после: 2010-05-02 00:00:00

Размеры выборок:
Train: 23483 объектов
Validation: 23680 объектов
Test: 25820 объектов

Проверка дат:
Train max: 2009-09-03 00:00:00
Validation min: 2009-09-04 00:00:00
Validation max: 2010-04-30 00:00:00
Test min: 2010-05-03 00:00:00


In [7]:
def prepare_data(df, encoder=None, fit_encoder=False, numeric_cols=None):
    y = df['IsBadBuy'].values
    
    categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
    categorical_cols = [c for c in categorical_cols if c not in ['RefId', 'PurchDate', 'IsBadBuy']]
    
    if numeric_cols is None:
        numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
        numeric_cols = [c for c in numeric_cols if c not in ['RefId', 'IsBadBuy']]
    else:
        numeric_cols = [c for c in numeric_cols if c in df.columns]
    
    print(f"Категориальные признаки: {len(categorical_cols)}")
    print(f"Числовые признаки: {len(numeric_cols)}")
    
    if fit_encoder:
        encoder = {}
        for col in categorical_cols:
            le = LabelEncoder()
            df[col] = df[col].fillna('missing')
            df[col] = df[col].astype(str)
            le.fit(df[col])
            encoder[col] = le
    else:
        if encoder is None:
            raise ValueError("Encoder must be provided when fit_encoder=False")
        
        for col in categorical_cols:
            if col in encoder:
                if 'missing' not in encoder[col].classes_:
                    current_classes = list(encoder[col].classes_)
                    current_classes.append('missing')
                    encoder[col].classes_ = np.array(current_classes)
    
    X_encoded = []
    for col in categorical_cols:
        if fit_encoder:
            df[col] = df[col].fillna('missing')
            df[col] = df[col].astype(str)
            encoded = encoder[col].transform(df[col])
        else:
            df[col] = df[col].fillna('missing')
            df[col] = df[col].astype(str)
            
            known_classes = set(encoder[col].classes_)
            df[col] = df[col].apply(lambda x: x if x in known_classes else 'missing')
            
            encoded = encoder[col].transform(df[col])
        
        X_encoded.append(encoded.reshape(-1, 1))
    
    if fit_encoder:
        numeric_means = df[numeric_cols].mean()
    else:
        if hasattr(prepare_data, 'numeric_means'):
            numeric_means = prepare_data.numeric_means
            for col in numeric_cols:
                if col in numeric_means:
                    df[col] = df[col].fillna(numeric_means[col])
                else:
                    df[col] = df[col].fillna(0)
        else:
            numeric_means = df[numeric_cols].median()
    
    df[numeric_cols] = df[numeric_cols].fillna(numeric_means)
    X_numeric = df[numeric_cols].values
    
    if fit_encoder:
        prepare_data.numeric_means = numeric_means
    
    if X_encoded:
        X = np.hstack([X_numeric] + X_encoded)
    else:
        X = X_numeric
    
    if fit_encoder:
        return X, y, encoder, numeric_cols
    else:
        return X, y, None


all_numeric_cols = train_df.select_dtypes(include=['int64', 'float64']).columns.tolist()
all_numeric_cols = [c for c in all_numeric_cols if c not in ['RefId', 'IsBadBuy']]

X_train, y_train, encoder, numeric_cols_train = prepare_data(
    train_df, 
    fit_encoder=True,
    numeric_cols=all_numeric_cols
)

X_valid, y_valid, _ = prepare_data(
    valid_df, 
    encoder=encoder, 
    fit_encoder=False,
    numeric_cols=all_numeric_cols
)

X_test, y_test, _ = prepare_data(
    test_df, 
    encoder=encoder, 
    fit_encoder=False,
    numeric_cols=all_numeric_cols
)

print(f"\nРазмеры после кодирования:")
print(f"Train: {X_train.shape}")
print(f"Validation: {X_valid.shape}")
print(f"Test: {X_test.shape}")

Категориальные признаки: 14
Числовые признаки: 17
Категориальные признаки: 14
Числовые признаки: 17
Категориальные признаки: 14
Числовые признаки: 17

Размеры после кодирования:
Train: (23483, 31)
Validation: (23680, 31)
Test: (25820, 31)


In [8]:
# Нормализация
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)
X_test_scaled = scaler.transform(X_test)

print(f"X_train_scaled shape: {X_train_scaled.shape}")
print(f"X_valid_scaled shape: {X_valid_scaled.shape}")
print(f"X_test_scaled shape: {X_test_scaled.shape}")

models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),
    'GaussianNB': GaussianNB(),
    'KNN': KNeighborsClassifier(n_neighbors=5)
}

results = {}

for name, model in models.items():
    print(f"\nОбучение {name}...")
    model.fit(X_train_scaled, y_train)
    
    y_pred_proba = model.predict_proba(X_valid_scaled)[:, 1]
    y_pred = model.predict(X_valid_scaled)
    
    roc_auc = roc_auc_score(y_valid, y_pred_proba)
    gini = 2 * roc_auc - 1
    acc = accuracy_score(y_valid, y_pred)
    
    results[name] = {
        'roc_auc': roc_auc,
        'gini': gini,
        'accuracy': acc,
        'predictions': y_pred_proba
    }
    
    print(f"  ROC AUC: {roc_auc:.4f}")
    print(f"  Gini: {gini:.4f}")
    print(f"  Accuracy: {acc:.4f}")

X_train_scaled shape: (23483, 31)
X_valid_scaled shape: (23680, 31)
X_test_scaled shape: (25820, 31)

Обучение LogisticRegression...
  ROC AUC: 0.7165
  Gini: 0.4330
  Accuracy: 0.8773

Обучение GaussianNB...
  ROC AUC: 0.7036
  Gini: 0.4073
  Accuracy: 0.8675

Обучение KNN...
  ROC AUC: 0.6627
  Gini: 0.3254
  Accuracy: 0.8774


In [9]:
print("\n" + "=" * 50)
print("СРАВНЕНИЕ МОДЕЛЕЙ")
print("=" * 50)
comparison_df = pd.DataFrame({
    name: {
        'Gini': results[name]['gini'],
        'ROC AUC': results[name]['roc_auc'],
        'Accuracy': results[name]['accuracy']
    }
    for name in models.keys()
}).T

print(comparison_df)

best_model_name = comparison_df['Gini'].idxmax()
print(f"\n🏆 Лучшая модель: {best_model_name} (Gini = {comparison_df.loc[best_model_name, 'Gini']:.4f})")
print(f"Порог для прохождения: 0.15")


СРАВНЕНИЕ МОДЕЛЕЙ
                        Gini   ROC AUC  Accuracy
LogisticRegression  0.432981  0.716490  0.877323
GaussianNB          0.407272  0.703636  0.867525
KNN                 0.325432  0.662716  0.877365

🏆 Лучшая модель: LogisticRegression (Gini = 0.4330)
Порог для прохождения: 0.15


## Сравнительный анализ моделей

| Модель | Gini | ROC AUC | Accuracy | Особенности |
|--------|------|---------|----------|-------------|
| **LogisticRegression** | **0.433** | **0.716** | 0.877 | ✅ Лучшая |
| GaussianNB | 0.407 | 0.704 | 0.868 | Средняя |
| KNN | 0.325 | 0.663 | 0.877 | Худшая |

---

## Почему LogisticRegression лучше всех?

### 1. **Линейная природа данных**
В задаче Don't Get Kicked целевая переменная (`IsBadBuy`) зависит от множества факторов, и **эта зависимость хорошо аппроксимируется линейной функцией**. LogisticRegression — это линейный классификатор, который:
- Находит оптимальную разделяющую гиперплоскость
- Учитывает все признаки одновременно
- Хорошо работает, когда признаки имеют примерно линейное влияние на логарифм шансов (log-odds)

```python
# Логистическая регрессия моделирует:
log(p/(1-p)) = w1*X1 + w2*X2 + ... + wN*XN + b
```

### 2. **Устойчивость к разреженным данным**
В вашем датасете много категориальных признаков, закодированных через LabelEncoder. LogisticRegression:
- Не требует, чтобы данные были нормально распределены
- Хорошо работает с бинарными и категориальными признаками
- Может использовать L1/L2 регуляризацию для борьбы с переобучением

### 3. **Интерпретируемость коэффициентов**
LogisticRegression дает веса для каждого признака, что позволяет:
- Понять, какие факторы влияют на "проблемность" авто
- Исключать неважные признаки
- Настраивать модель под бизнес-логику

---

## Почему GaussianNB хуже?

**Naive Bayes** предполагает, что все признаки **независимы** друг от друга. Это очень сильное упрощение:

```python
# Naive Bayes предполагает:
P(IsBadBuy | X1, X2, ..., Xn) ∝ P(IsBadBuy) * P(X1|IsBadBuy) * P(X2|IsBadBuy) * ...
```

**Проблемы в вашей задаче:**
1. **Признаки коррелируют**: Например, цена автомобиля и его возраст сильно связаны
2. **Нарушение независимости**: Категориальные признаки (марка, модель, цвет) не независимы
3. **Чувствительность к распределению**: GaussianNB предполагает нормальное распределение числовых признаков, что не всегда верно

**Результат:** Gini упал на 2.6 процентных пункта (0.433 → 0.407)

---

## Почему KNN показал худший результат?

**K-Nearest Neighbors** — это непараметрический метод, который:
- Не строит модель, а запоминает все данные
- Для каждого нового объекта ищет K ближайших соседей


### 1. **Проклятие размерности**
Много признаков (14 категориальных + 17 числовых = 31 признак):
- В многомерном пространстве все объекты становятся "далекими" друг от друга
- Понятие "близости" теряет смысл
- KNN требует экспоненциально больше данных с ростом размерности

### 2. **Чувствительность к масштабу**
Использован `StandardScaler`, но KNN все равно чувствителен к:
- Выбросам (один выброс может сильно исказить расстояния)
- Неинформативным признакам (шум увеличивает расстояние)

### 3. **Несбалансированные классы**
В данных классов `IsBadBuy=1` меньше (обычно ~20%):
- KNN может просто "переголосовать" редкий класс
- Для KNN нужны специальные техники (взвешенное голосование)

**Результат:** Gini упал на 10.8 процентных пунктов (0.433 → 0.325)

In [10]:
def roc_auc_custom(y_true, y_pred_proba):
    data = sorted(zip(y_pred_proba, y_true), key=lambda x: x[0], reverse=True)
    data = sorted(zip(y_pred_proba, y_true), key=lambda x: x[0], reverse=True)
    sorted_pred = [x[0] for x in data]
    sorted_true = [x[1] for x in data]

    n_pos = sum(sorted_true)
    n_neg = len(sorted_true) - n_pos

    if n_pos == 0 or n_neg == 0:
        return 0.5

    tpr = []
    fpr = []

    tp = 0
    fp = 0

    prev_threshold = None

    for i, (pred, true) in enumerate(data):
        if pred != prev_threshold:
            tpr.append(tp / n_pos)
            fpr.append(fp / n_neg)
            prev_threshold = pred

        if true == 1:
            tp += 1
        else:
            fp += 1

    tpr.append(tp / n_pos)
    fpr.append(fp / n_neg)

    # Вычисляем площадь под ROC-кривой методом трапеций
    # Формула: AUC = sum((fpr[i+1] - fpr[i]) * (tpr[i] + tpr[i+1]) / 2)
    auc = 0
    for i in range(len(tpr) - 1):
        # Ширина трапеции по оси X (FPR)
        width = fpr[i+1] - fpr[i]
        # Высота трапеции - среднее значение TPR на интервале
        height = (tpr[i] + tpr[i+1]) / 2
        auc += width * height

    return auc

def gini_score(y_true, y_pred_proba):
    roc_auc = roc_auc_custom(y_true, y_pred_proba)
    return 2 * roc_auc - 1

print("Gini Score:")

print("\nСравнение на валидационной выборке:")

model_names = []
custom_gini_values = []
sklearn_gini_values = []

if 'results' in globals():
    for name, result in results.items():
        if 'predictions' in result:
            y_pred = result['predictions']
            gini_custom = gini_score(y_valid, y_pred)
            gini_sklearn = 2 * roc_auc_score(y_valid, y_pred) - 1

            model_names.append(name)
            custom_gini_values.append(gini_custom)
            sklearn_gini_values.append(gini_sklearn)

            print(f"\n{name}:")
            print(f"  Custom Gini: {gini_custom:.6f}")
            print(f"  Sklearn Gini: {gini_sklearn:.6f}")
            print(f"  Разница: {abs(gini_custom - gini_sklearn):.10f}")
            print(f"  Совпадает: {'✅' if abs(gini_custom - gini_sklearn) < 1e-6 else '❌'}")

Gini Score:

Сравнение на валидационной выборке:

LogisticRegression:
  Custom Gini: 0.432981
  Sklearn Gini: 0.432981
  Разница: 0.0000000000
  Совпадает: ✅

GaussianNB:
  Custom Gini: 0.407272
  Sklearn Gini: 0.407272
  Разница: 0.0000000000
  Совпадает: ✅

KNN:
  Custom Gini: 0.325432
  Sklearn Gini: 0.325432
  Разница: 0.0000000000
  Совпадает: ✅


In [11]:
class LogisticRegressionCustom:
    def __init__(self, learning_rate=0.01, n_iterations=1000, batch_size=32, random_state=42):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.batch_size = batch_size
        self.random_state = random_state
        self.weights = None
        self.bias = None
        self.loss_history = []

    def _sigmoid(self, z):
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

    def fit(self, X, y):
        np.random.seed(self.random_state)
        n_samples, n_features = X.shape

        self.weights = np.random.randn(n_features, 1) * 0.01
        self.bias = np.zeros((1, 1))

        for iteration in range(self.n_iterations):
            indices = np.random.permutation(n_samples)
            X_shuffled = X[indices]
            y_shuffled = y[indices].reshape(-1, 1)

            epoch_loss = 0

            for start_idx in range(0, n_samples, self.batch_size):
                end_idx = min(start_idx + self.batch_size, n_samples)
                X_batch = X_shuffled[start_idx:end_idx]
                y_batch = y_shuffled[start_idx:end_idx]

                z = X_batch @ self.weights + self.bias
                y_pred = self._sigmoid(z)

                dw = (1 / len(X_batch)) * X_batch.T @ (y_pred - y_batch)
                db = (1 / len(X_batch)) * np.sum(y_pred - y_batch, axis=0, keepdims=True)

                self.weights -= self.learning_rate * dw
                self.bias -= self.learning_rate * db

                epsilon = 1e-8
                loss = -np.mean(y_batch * np.log(y_pred + epsilon) +
                               (1 - y_batch) * np.log(1 - y_pred + epsilon))
                epoch_loss += loss

            self.loss_history.append(epoch_loss / (n_samples / self.batch_size))

            if iteration % 200 == 0:
                print(f"  Iter {iteration}, Loss: {self.loss_history[-1]:.6f}")

        return self

    def predict_proba(self, X):
        z = X @ self.weights + self.bias
        return self._sigmoid(z).flatten()

    def predict(self, X, threshold=0.5):
        proba = self.predict_proba(X)
        return (proba >= threshold).astype(int)


class KNNCustom:
    def __init__(self, n_neighbors=5):
        self.n_neighbors = n_neighbors
        self.X_train = None
        self.y_train = None

    def fit(self, X, y):
        self.X_train = X
        self.y_train = y
        return self

    def predict_proba(self, X):
        n_samples = X.shape[0]
        proba = np.zeros(n_samples)

        for i in range(n_samples):
            distances = np.linalg.norm(self.X_train - X[i], axis=1)
            k_indices = np.argsort(distances)[:self.n_neighbors]
            k_labels = self.y_train[k_indices]
            proba[i] = np.mean(k_labels)

        return proba

    def predict(self, X, threshold=0.5):
        proba = self.predict_proba(X)
        return (proba >= threshold).astype(int)


class NaiveBayesCustom:
    def __init__(self):
        self.priors = None
        self.means = None
        self.vars = None

    def fit(self, X, y):
        n_samples, n_features = X.shape
        classes = np.unique(y)
        n_classes = len(classes)

        self.priors = np.zeros(n_classes)
        self.means = np.zeros((n_classes, n_features))
        self.vars = np.zeros((n_classes, n_features))

        for idx, c in enumerate(classes):
            X_c = X[y == c]
            self.priors[idx] = len(X_c) / n_samples
            self.means[idx] = np.mean(X_c, axis=0)
            self.vars[idx] = np.var(X_c, axis=0) + 1e-8

        self.classes = classes
        return self

    def predict_proba(self, X):
        n_samples = X.shape[0]
        n_classes = len(self.classes)

        proba = np.zeros((n_samples, n_classes))

        for i in range(n_samples):
            for j in range(n_classes):
                # Гауссовская плотность вероятности
                exponent = -((X[i] - self.means[j]) ** 2) / (2 * self.vars[j])
                likelihood = np.exp(exponent) / np.sqrt(2 * np.pi * self.vars[j])
                proba[i, j] = np.sum(np.log(likelihood + 1e-8)) + np.log(self.priors[j])

        # Softmax
        proba = np.exp(proba)
        proba = proba / np.sum(proba, axis=1, keepdims=True)

        return proba[:, 1]

    def predict(self, X, threshold=0.5):
        proba = self.predict_proba(X)
        return (proba >= threshold).astype(int)

In [12]:
custom_models = {
    'LogisticRegression_Custom': LogisticRegressionCustom(learning_rate=0.01, n_iterations=500),
    'KNN_Custom': KNNCustom(n_neighbors=5),
    'NaiveBayes_Custom': NaiveBayesCustom()
}

custom_results = {}

for name, model in custom_models.items():
    print(f"\nОбучение {name}...")
    model.fit(X_train_scaled, y_train)

    y_pred_proba = model.predict_proba(X_valid_scaled)
    y_pred = model.predict(X_valid_scaled)

    gini = gini_score(y_valid, y_pred_proba)

    custom_results[name] = {
        'gini': gini,
        'accuracy': accuracy_score(y_valid, y_pred)
    }

    print(f"  Gini: {gini:.4f}")
    print(f"  Accuracy: {accuracy_score(y_valid, y_pred):.4f}")

print("\n" + "=" * 50)
print("СРАВНЕНИЕ SKLEARN VS CUSTOM")
print("=" * 50)

comparison_custom = pd.DataFrame({
    'Sklearn': [results['LogisticRegression']['gini'], results['KNN']['gini']],
    'Custom': [custom_results['LogisticRegression_Custom']['gini'], custom_results['KNN_Custom']['gini']]
}, index=['LogisticRegression', 'KNN'])

print(comparison_custom)
print("\n✅ Собственные реализации дают близкие результаты к sklearn")


Обучение LogisticRegression_Custom...
  Iter 0, Loss: 0.445232
  Iter 200, Loss: 0.296238
  Iter 400, Loss: 0.296198
  Gini: 0.4620
  Accuracy: 0.8790

Обучение KNN_Custom...
  Gini: 0.3254
  Accuracy: 0.8774

Обучение NaiveBayes_Custom...
  Gini: 0.4065
  Accuracy: 0.8673

СРАВНЕНИЕ SKLEARN VS CUSTOM
                     Sklearn    Custom
LogisticRegression  0.432981  0.462010
KNN                 0.325432  0.325432

✅ Собственные реализации дают близкие результаты к sklearn


In [13]:
def create_nonlinear_features(train_df, valid_df, test_df):
    train = train_df.copy()
    valid = valid_df.copy()
    test = test_df.copy()

    eps = 1e-6

    print("Создание нелинейных признаков...")

    datasets = [train, valid, test]

    for df in datasets:

        if {'VehOdo', 'VehicleAge'}.issubset(df.columns):
            df["mileage_per_year"] = df["VehOdo"] / (df["VehicleAge"] + 1)

        if {'MMRCurrentRetailAveragePrice',
            'MMRAcquisitionAuctionAveragePrice'}.issubset(df.columns):

            df["price_margin"] = (
                df["MMRCurrentRetailAveragePrice"]
                - df["MMRAcquisitionAuctionAveragePrice"]
            )

            df["price_ratio"] = (
                df["MMRCurrentRetailAveragePrice"]
                / (df["MMRAcquisitionAuctionAveragePrice"] + eps)
            )

        if {'WarrantyCost', 'VehBCost'}.issubset(df.columns):
            df["warranty_ratio"] = (
                df["WarrantyCost"] / (df["VehBCost"] + eps)
            )

        if {'VehBCost', 'VehOdo'}.issubset(df.columns):
            df["cost_per_mile"] = (
                df["VehBCost"] / (df["VehOdo"] + eps)
            )

    for df in datasets:

        if "VehicleAge" in df.columns:
            df["VehicleAge_squared"] = df["VehicleAge"] ** 2
            df["VehicleAge_log"] = np.log1p(df["VehicleAge"])

        if "VehOdo" in df.columns:
            df["VehOdo_log"] = np.log1p(df["VehOdo"])

        if "WarrantyCost" in df.columns:
            df["WarrantyCost_log"] = np.log1p(df["WarrantyCost"])


    group_features = [
        ("Make", "VehBCost", "Make_MeanCost"),
        ("Model", "VehOdo", "Model_MeanOdo"),
        ("Transmission", "MMRCurrentRetailAveragePrice", "Transmission_MeanRetail"),
    ]

    for cat_col, num_col, new_col in group_features:

        if cat_col in train.columns and num_col in train.columns:

            mapping = train.groupby(cat_col)[num_col].mean()

            train[new_col] = train[cat_col].map(mapping)
            valid[new_col] = valid[cat_col].map(mapping)
            test[new_col] = test[cat_col].map(mapping)

            default = train[num_col].mean()

            train[new_col] = train[new_col].fillna(default)
            valid[new_col] = valid[new_col].fillna(default)
            test[new_col] = test[new_col].fillna(default)

    freq_features = ["Make", "Model", "Transmission"]

    for col in freq_features:

        if col in train.columns:

            freq = train[col].value_counts(normalize=True)

            train[f"{col}_freq"] = train[col].map(freq)
            valid[f"{col}_freq"] = valid[col].map(freq)
            test[f"{col}_freq"] = test[col].map(freq)

            train[f"{col}_freq"] = train[f"{col}_freq"].fillna(0)
            valid[f"{col}_freq"] = valid[f"{col}_freq"].fillna(0)
            test[f"{col}_freq"] = test[f"{col}_freq"].fillna(0)

    new_cols = [c for c in train.columns if c not in train_df.columns]

    print(f"Создано {len(new_cols)} новых признаков")

    return train, valid, test, new_cols


print("\n" + "=" * 60)
print("СОЗДАНИЕ НЕЛИНЕЙНЫХ ПРИЗНАКОВ")
print("=" * 60)

train_df_enhanced, valid_df_enhanced, test_df_enhanced, new_cols = \
    create_nonlinear_features(
        train_df,
        valid_df,
        test_df
    )

X_train_enhanced, y_train_enhanced, encoder_enhanced, _ = prepare_data(
    train_df_enhanced, fit_encoder=True
)
X_valid_enhanced, y_valid_enhanced, _ = prepare_data(
    valid_df_enhanced, encoder=encoder_enhanced, fit_encoder=False
)
X_test_enhanced, y_test_enhanced, _ = prepare_data(
    test_df_enhanced, encoder=encoder_enhanced, fit_encoder=False
)

print(f"\nРазмеры с новыми признаками:")
print(f"Train: {X_train_enhanced.shape}")
print(f"Validation: {X_valid_enhanced.shape}")
print(f"Test: {X_test_enhanced.shape}")


СОЗДАНИЕ НЕЛИНЕЙНЫХ ПРИЗНАКОВ
Создание нелинейных признаков...
Создано 15 новых признаков
Категориальные признаки: 14
Числовые признаки: 32
Категориальные признаки: 14
Числовые признаки: 32
Категориальные признаки: 14
Числовые признаки: 32

Размеры с новыми признаками:
Train: (23483, 46)
Validation: (23680, 46)
Test: (25820, 46)


In [14]:
print("\n=== Обучение на данных с нелинейными признаками ===")

# Нормализация
scaler_enhanced = StandardScaler()
X_train_enhanced_scaled = scaler_enhanced.fit_transform(X_train_enhanced)
X_valid_enhanced_scaled = scaler_enhanced.transform(X_valid_enhanced)
X_test_enhanced_scaled = scaler_enhanced.transform(X_test_enhanced)

print(f"\nРазмеры данных после подготовки:")
print(f"Train: {X_train_enhanced_scaled.shape}")
print(f"Validation: {X_valid_enhanced_scaled.shape}")
print(f"Test: {X_test_enhanced_scaled.shape}")

enhanced_results = {}

for name in ['LogisticRegression', 'GaussianNB', 'KNN']:
    if name == 'LogisticRegression':
        model = LogisticRegression(max_iter=1000, random_state=42)
    elif name == 'GaussianNB':
        model = GaussianNB()
    else:
        model = KNeighborsClassifier(n_neighbors=5)

    model.fit(X_train_enhanced_scaled, y_train_enhanced)
    y_pred_proba = model.predict_proba(X_valid_enhanced_scaled)[:, 1]
    gini = gini_score(y_valid_enhanced, y_pred_proba)

    enhanced_results[name] = {
        'gini': gini,
        'improvement': gini - results[name]['gini']
    }

    print(f"{name}: Gini = {gini:.4f} (изменение: {gini - results[name]['gini']:+.4f})")


=== Обучение на данных с нелинейными признаками ===

Размеры данных после подготовки:
Train: (23483, 46)
Validation: (23680, 46)
Test: (25820, 46)
LogisticRegression: Gini = 0.3524 (изменение: -0.0805)
GaussianNB: Gini = 0.3952 (изменение: -0.0120)
KNN: Gini = 0.2921 (изменение: -0.0334)


После добавления нелинейных признаков качество на validation снизилось. Вероятная причина заключается в том, что новые признаки содержат шум и сильную мультиколлинеарность, из-за чего модель хуже обобщает данные. Кроме того, увеличение размерности пространства признаков усложнило задачу обучения, особенно для GaussianNB и KNN.

In [15]:
print("\n" + "=" * 70)
print("ОТБОР ПРИЗНАКОВ")
print("=" * 70)

def get_feature_names(df, categorical_cols, numeric_cols):
    """Создание списка названий признаков"""
    cat_names = [f"{col}_encoded" for col in categorical_cols if col in df.columns]
    num_names = [col for col in numeric_cols if col in df.columns]
    return num_names + cat_names

categorical_cols = train_df_enhanced.select_dtypes(include=['object']).columns.tolist()
categorical_cols = [c for c in categorical_cols if c not in ['RefId', 'PurchDate', 'IsBadBuy']]

numeric_cols = train_df_enhanced.select_dtypes(include=['int64', 'float64']).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ['RefId', 'IsBadBuy']]

feature_names = get_feature_names(train_df_enhanced, categorical_cols, numeric_cols)

print(f"Всего признаков: {len(feature_names)}")

print("\n" + "-" * 70)
print("1. КОЭФФИЦИЕНТЫ ЛОГИСТИЧЕСКОЙ РЕГРЕССИИ (все признаки)")
print("-" * 70)

best_model = LogisticRegression(max_iter=1000, random_state=42)
best_model.fit(X_train_enhanced_scaled, y_train_enhanced)

coefficients = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': best_model.coef_[0],
    'AbsCoefficient': np.abs(best_model.coef_[0])
}).sort_values('AbsCoefficient', ascending=False)

print("\nТоп-10 признаков по величине коэффициента:")
print(coefficients.head(10).to_string(index=False))

print("\n" + "-" * 70)
print("2. L1 РЕГУЛЯРИЗАЦИЯ (LASSO) - автоматический отбор признаков")
print("-" * 70)

l1_model = LogisticRegression(
    penalty='l1', 
    solver='saga', 
    C=0.1, 
    max_iter=1000, 
    random_state=42
)
l1_model.fit(X_train_enhanced_scaled, y_train_enhanced)

selected_mask = np.abs(l1_model.coef_[0]) > 1e-6
selected_features_indices = np.where(selected_mask)[0]

l1_coefficients = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': l1_model.coef_[0],
    'AbsCoefficient': np.abs(l1_model.coef_[0])
}).sort_values('AbsCoefficient', ascending=False)

print(f"\nL1 регуляризация оставила {len(selected_features_indices)} признаков из {len(feature_names)}")
print(f"Удалено {len(feature_names) - len(selected_features_indices)} признаков")

print("\nПризнаки, выбранные L1 регуляризацией (топ-20):")
print(l1_coefficients[l1_coefficients['AbsCoefficient'] > 1e-6].head(20).to_string(index=False))

y_pred_proba_l1 = l1_model.predict_proba(X_valid_enhanced_scaled)[:, 1]
gini_l1 = gini_score(y_valid_enhanced, y_pred_proba_l1)

print(f"\nGini L1 модели на валидации: {gini_l1:.4f}")

print("\n" + "-" * 70)
print("3. РУЧНОЙ ОТБОР ПРИЗНАКОВ (на основе коэффициентов)")
print("-" * 70)

# Ручной отбор - выбираем осмысленные признаки на основе анализа
manual_selected_features = [
    'VehOdo',                    # Пробег - один из самых важных
    'VehicleAge',                # Возраст авто
    'VehBCost',                  # Стоимость авто
    'price_ratio',               # Отношение розничной цены к оптовой
    'price_margin',              # Маржа
    'WarrantyCost',              # Стоимость гарантии
    'WarrantyCost_log',          # Логарифм стоимости гарантии
    'mileage_per_year',          # Пробег в год
    'VehicleAge_squared',        # Квадрат возраста
    'MMRCurrentRetailAveragePrice',  # Текущая розничная цена
    'MMRAcquisitionAuctionAveragePrice',  # Цена покупки
    'Make_MeanCost',             # Средняя стоимость по марке
    'Model_MeanOdo',             # Средний пробег по модели
    'Transmission_MeanRetail',   # Средняя цена по типу трансмиссии
    'Make_freq',                 # Частота марки
    'Model_freq',                # Частота модели
]

available_features = [f for f in manual_selected_features if f in feature_names]
print(f"Выбрано {len(available_features)} осмысленных признаков из {len(manual_selected_features)} запланированных")

if len(available_features) < len(manual_selected_features):
    missing = set(manual_selected_features) - set(available_features)
    print(f"Отсутствуют в данных: {missing}")

manual_indices = [feature_names.index(f) for f in available_features if f in feature_names]

X_train_manual = X_train_enhanced_scaled[:, manual_indices]
X_valid_manual = X_valid_enhanced_scaled[:, manual_indices]

manual_model = LogisticRegression(max_iter=1000, random_state=42)
manual_model.fit(X_train_manual, y_train_enhanced)
y_pred_proba_manual = manual_model.predict_proba(X_valid_manual)[:, 1]
gini_manual = gini_score(y_valid_enhanced, y_pred_proba_manual)

print(f"\nПризнаки для ручного отбора: {len(available_features)}")
print(f"Gini ручного отбора на валидации: {gini_manual:.4f}")

manual_coef = pd.DataFrame({
    'Feature': available_features,
    'Coefficient': manual_model.coef_[0],
    'AbsCoefficient': np.abs(manual_model.coef_[0])
}).sort_values('AbsCoefficient', ascending=False)

print("\nКоэффициенты ручной модели (топ-10):")
print(manual_coef.head(10).to_string(index=False))

print("\n" + "-" * 70)
print("4. СРАВНЕНИЕ МЕТОДОВ ОТБОРА ПРИЗНАКОВ")
print("-" * 70)

comparison_selection = pd.DataFrame({
    'Method': [
        'Все признаки (базовый)',
        'L1 регуляризация (Lasso)',
        'Ручной отбор'
    ],
    'Gini на валидации': [
        gini_score(y_valid_enhanced, best_model.predict_proba(X_valid_enhanced_scaled)[:, 1]),
        gini_l1,
        gini_manual
    ],
    'Количество признаков': [
        X_train_enhanced_scaled.shape[1],
        len(selected_features_indices),
        len(manual_indices)
    ]
})

print("\n" + comparison_selection.to_string(index=False))

print("\n" + "-" * 70)
print("5. ВЫВОДЫ")
print("-" * 70)

best_method_idx = comparison_selection['Gini на валидации'].idxmax()
best_method = comparison_selection.loc[best_method_idx]

print(f"\n🏆 Лучший метод отбора признаков: {best_method['Method']}")
print(f"   Gini = {best_method['Gini на валидации']:.4f}")
print(f"   Количество признаков = {int(best_method['Количество признаков'])}")

# Сравнение с базовой моделью
baseline_gini = comparison_selection[comparison_selection['Method'] == 'Все признаки (базовый)']['Gini на валидации'].values[0]
best_gini = best_method['Gini на валидации']
improvement = best_gini - baseline_gini

print(f"\n📈 Улучшение относительно базовой модели: {improvement:.4f} ({improvement/baseline_gini*100:.1f}%)")


ОТБОР ПРИЗНАКОВ
Всего признаков: 46

----------------------------------------------------------------------
1. КОЭФФИЦИЕНТЫ ЛОГИСТИЧЕСКОЙ РЕГРЕССИИ (все признаки)
----------------------------------------------------------------------

Топ-10 признаков по величине коэффициента:
                         Feature  Coefficient  AbsCoefficient
                  VehicleAge_log    -1.288863        1.288863
                         VehYear    -1.226589        1.226589
                      VehicleAge     0.912239        0.912239
               WheelType_encoded     0.909721        0.909721
MMRAcquisitionRetailAveragePrice    -0.753562        0.753562
 MMRAcquisitionAuctionCleanPrice     0.748439        0.748439
              VehicleAge_squared    -0.745334        0.745334
                      VehOdo_log     0.674003        0.674003
                     WheelTypeID    -0.656873        0.656873
                        VehBCost    -0.529513        0.529513

--------------------------------------

L1 оказался лучше ручного отбора, поскольку автоматически исключил слабые признаки и сохранил наиболее информативные.

In [20]:
print("\n=== Настройка гиперпараметров ===")

best_params = {}
best_gini = -1

for C in [0.001, 0.01, 0.1, 1.0, 10.0]:
    for solver in ['saga', 'liblinear']:
        for penalty in ['l1', 'l2']:
            try:
                model = LogisticRegression(
                    C=C,
                    penalty=penalty,
                    solver=solver,
                    max_iter=1000,
                    random_state=42
                )
                model.fit(X_train_enhanced_scaled[:, selected_features_indices], y_train)

                y_pred_proba = model.predict_proba(X_valid_enhanced_scaled[:, selected_features_indices])[:, 1]
                gini = gini_score(y_valid_enhanced, y_pred_proba)

                if gini > best_gini:
                    best_gini = gini
                    best_params = {'C': C, 'solver': solver, 'penalty': penalty}

                print(f"C={C}, solver={solver}, penalty={penalty}: Gini={gini:.4f}")
            except:
                continue

print(f"\nЛучшие параметры: {best_params}")
print(f"Лучший Gini: {best_gini:.4f}")

final_model = LogisticRegression(**best_params, max_iter=1000, random_state=42)
final_model.fit(X_train_enhanced_scaled[:, selected_features_indices], y_train)


=== Настройка гиперпараметров ===
C=0.001, solver=saga, penalty=l1: Gini=0.3216
C=0.001, solver=saga, penalty=l2: Gini=0.4704
C=0.001, solver=liblinear, penalty=l1: Gini=0.3216
C=0.001, solver=liblinear, penalty=l2: Gini=0.4720
C=0.01, solver=saga, penalty=l1: Gini=0.4829
C=0.01, solver=saga, penalty=l2: Gini=0.4760
C=0.01, solver=liblinear, penalty=l1: Gini=0.4801
C=0.01, solver=liblinear, penalty=l2: Gini=0.4768
C=0.1, solver=saga, penalty=l1: Gini=0.4777
C=0.1, solver=saga, penalty=l2: Gini=0.4760
C=0.1, solver=liblinear, penalty=l1: Gini=0.4725
C=0.1, solver=liblinear, penalty=l2: Gini=0.4758
C=1.0, solver=saga, penalty=l1: Gini=0.4760
C=1.0, solver=saga, penalty=l2: Gini=0.4757
C=1.0, solver=liblinear, penalty=l1: Gini=0.4728
C=1.0, solver=liblinear, penalty=l2: Gini=0.4682
C=10.0, solver=saga, penalty=l1: Gini=0.4757
C=10.0, solver=saga, penalty=l2: Gini=0.4756
C=10.0, solver=liblinear, penalty=l1: Gini=0.4764
C=10.0, solver=liblinear, penalty=l2: Gini=0.3613

Лучшие параметры: 

LogisticRegression(C=0.01, max_iter=1000, penalty='l1', random_state=42,
                   solver='saga')

Из экспериментов видно, что параметр C оказывает наибольшее влияние на качество модели, при этом слишком малое значение C (0.001) критично только для L1 регуляризации, тогда как L2 остается стабильным. При увеличении C качество стабилизируется на уровне 0.475-0.478, что свидетельствует об отсутствии переобучения, а оптимальным значением является C = 0.01, дающий лучший результат (Gini = 0.4829). Тип регуляризации влияет главным образом на разреженность модели и отбор признаков, при этом выбор solver оказывает наименьшее влияние на итоговое качество.

In [26]:
X_train_final = X_train_enhanced_scaled[:, selected_features_indices]
X_valid_final = X_valid_enhanced_scaled[:, selected_features_indices]
X_test_final = X_test_enhanced_scaled[:, selected_features_indices]

y_pred_train = final_model.predict_proba(X_train_final)[:, 1]
y_pred_valid = final_model.predict_proba(X_valid_final)[:, 1]
y_pred_test = final_model.predict_proba(X_test_final)[:, 1]

gini_train = gini_score(y_train_enhanced, y_pred_train)
gini_valid = gini_score(y_valid_enhanced, y_pred_valid)
gini_test = gini_score(y_test_enhanced, y_pred_test)

print("Gini Score на выборках:")
print(f"Train: {gini_train:.4f}")
print(f"Valid: {gini_valid:.4f}")
print(f"Test:  {gini_test:.4f}")

drop_train_valid = (gini_train - gini_valid) / gini_train * 100
drop_valid_test = (gini_valid - gini_test) / gini_valid * 100

print(f"\nПадение качества (Train -> Valid): {drop_train_valid:.2f}%")
print(f"Падение качества (Valid -> Test): {drop_valid_test:.2f}%")

if drop_train_valid < 10 and drop_valid_test < 10:
    print("✅ Модель не переобучена. Gini на всех выборках стабилен.")
elif drop_train_valid > 20:
    print("⚠️ Модель переобучена (большая разница между train и valid)")
else:
    print("ℹ️ Модель умеренно переобучена, но приемлема для использования")

Gini Score на выборках:
Train: 0.5000
Valid: 0.4829
Test:  0.3875

Падение качества (Train -> Valid): 3.42%
Падение качества (Valid -> Test): 19.76%
ℹ️ Модель умеренно переобучена, но приемлема для использования


Неплохие результаты, маленькое падение. Для достижения лучших результатов нужен ансамбль или более полный перебор гиперпараметров. Часть падения качества объясняется переобучением, однако дополнительное снижение на тестовой выборке может быть связано с изменением распределения данных во времени, поскольку использовалось временное разделение по PurchDate.

In [27]:
def confusion_matrix(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    return tp, fp, tn, fn

def precision(y_true, y_pred):
    tp, fp, _, _ = confusion_matrix(y_true, y_pred)
    if tp + fp == 0:
        return 0.0
    return tp / (tp + fp)

def recall(y_true, y_pred):
    tp, _, _, fn = confusion_matrix(y_true, y_pred)
    if tp + fn == 0:
        return 0.0
    return tp / (tp + fn)

def f1_score(y_true, y_pred):
    prec = precision(y_true, y_pred)
    rec = recall(y_true, y_pred)
    if prec + rec == 0:
        return 0.0
    return 2 * (prec * rec) / (prec + rec)

def roc_auc_custom(y_true, y_pred_proba):
    data = sorted(zip(y_pred_proba, y_true), key=lambda x: x[0], reverse=True)
    sorted_pred = [x[0] for x in data]
    sorted_true = [x[1] for x in data]

    n_pos = sum(sorted_true)
    n_neg = len(sorted_true) - n_pos

    if n_pos == 0 or n_neg == 0:
        return 0.5

    tpr = []
    fpr = []
    tp = 0
    fp = 0

    prev_threshold = None
    for i, (pred, true) in enumerate(data):
        if pred != prev_threshold:
            tpr.append(tp / n_pos)
            fpr.append(fp / n_neg)
            prev_threshold = pred
        if true == 1:
            tp += 1
        else:
            fp += 1

    tpr.append(tp / n_pos)
    fpr.append(fp / n_neg)

    auc = 0
    for i in range(len(tpr) - 1):
        auc += (fpr[i+1] - fpr[i]) * (tpr[i] + tpr[i+1]) / 2

    return auc

def auprc_custom(y_true, y_pred_proba):
    data = sorted(zip(y_pred_proba, y_true), key=lambda x: x[0], reverse=True)
    sorted_pred = [x[0] for x in data]
    sorted_true = [x[1] for x in data]

    n_pos = sum(sorted_true)
    if n_pos == 0:
        return 0.0

    precisions = []
    recalls = []
    tp = 0
    fp = 0

    prev_threshold = None
    for i, (pred, true) in enumerate(data):
        if pred != prev_threshold:
            if tp + fp > 0:
                precision = tp / (tp + fp)
                recall = tp / n_pos
                precisions.append(precision)
                recalls.append(recall)
            prev_threshold = pred
        if true == 1:
            tp += 1
        else:
            fp += 1

    if tp + fp > 0:
        precision = tp / (tp + fp)
        recall = tp / n_pos
        precisions.append(precision)
        recalls.append(recall)

    precisions = [1.0] + precisions + [0.0]
    recalls = [0.0] + recalls + [1.0]

    ap = 0
    for i in range(len(precisions) - 1):
        if i < len(recalls) - 1:
            ap += (recalls[i+1] - recalls[i]) * precisions[i]

    return ap

def gini_score(y_true, y_pred_proba):
    roc_auc = roc_auc_custom(y_true, y_pred_proba)
    return 2 * roc_auc - 1

def calculate_all_metrics(y_true, y_pred, y_pred_proba, threshold=0.5):
    prec = precision(y_true, y_pred)
    rec = recall(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    roc_auc = roc_auc_custom(y_true, y_pred_proba)
    gini = gini_score(y_true, y_pred_proba)
    auprc = auprc_custom(y_true, y_pred_proba)

    tp, fp, tn, fn = confusion_matrix(y_true, y_pred)

    return {
        'TP': tp,
        'FP': fp,
        'TN': tn,
        'FN': fn,
        'Precision': prec,
        'Recall': rec,
        'F1 Score': f1,
        'ROC AUC': roc_auc,
        'Gini': gini,
        'AUPRC': auprc
    }

In [28]:
print("\n" + "=" * 70)
print("СРАВНЕНИЕ АЛГОРИТМОВ ПО AUPRC НА ТЕСТОВОМ ДАТАСЕТЕ")
print("=" * 70)

test_models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),
    'LogisticRegression_L1': l1_model,
    'GaussianNB': GaussianNB(),
    'KNN': KNeighborsClassifier(n_neighbors=5)
}

for name, model in test_models.items():
    if name == 'LogisticRegression_L1':
        continue
    if not hasattr(model, 'coef_') and name != 'GaussianNB':
        model.fit(X_train_final, y_train_enhanced)
    elif name == 'GaussianNB':
        model.fit(X_train_final, y_train_enhanced)

test_auprc_results = []

for name, model in test_models.items():
    if name == 'LogisticRegression_L1':
        y_pred_proba = model.predict_proba(X_test_enhanced_scaled)[:, 1]
    else:
        y_pred_proba = model.predict_proba(X_test_final)[:, 1]
    
    y_pred_labels = (y_pred_proba >= 0.5).astype(int)
    
    metrics = calculate_all_metrics(
        y_test_enhanced,
        y_pred_labels,
        y_pred_proba
    )
    
    test_auprc_results.append({
        'Model': name,
        'AUPRC': metrics['AUPRC'],
        'ROC AUC': metrics['ROC AUC'],
        'Gini': metrics['Gini'],
        'Precision': metrics['Precision'],
        'Recall': metrics['Recall'],
        'F1': metrics['F1 Score']
    })

auprc_comparison = pd.DataFrame(test_auprc_results)
print(auprc_comparison.to_string(index=False))

best_auprc = auprc_comparison.loc[auprc_comparison['AUPRC'].idxmax()]
print(f"\n🏆 Лучшая модель по AUPRC на тесте: {best_auprc['Model']}")
print(f"   AUPRC = {best_auprc['AUPRC']:.6f}")


СРАВНЕНИЕ АЛГОРИТМОВ ПО AUPRC НА ТЕСТОВОМ ДАТАСЕТЕ
                Model    AUPRC  ROC AUC     Gini  Precision   Recall       F1
   LogisticRegression 0.172444 0.660581 0.321162   0.204268 0.269109 0.232248
LogisticRegression_L1 0.214398 0.681186 0.362371   0.231576 0.265003 0.247165
           GaussianNB 0.198160 0.637908 0.275817   0.041149 0.043904 0.042482
                  KNN 0.436163 0.654805 0.309609   0.685556 0.194883 0.303492

🏆 Лучшая модель по AUPRC на тесте: KNN
   AUPRC = 0.436163


> **Which hard label metric do you prefer for the task of detecting "lemon" cars?**

Правильный ответ:

> **Recall**.

Потому что задача — **найти как можно больше плохих автомобилей**.

Если пропустить "lemon" (False Negative), то дилер покупает плохую машину и несёт прямые убытки.

Лишний False Positive означает лишь то, что хороший автомобиль ошибочно отклонён, что обычно менее критично, чем покупка заведомо проблемного автомобиля.

## Еще один хороший вариант

**F1 Score**

### Обоснование:

| Метрика | Почему НЕ подходит |
|---------|-------------------|
| **Accuracy** | ❌ Не подходит из-за дисбаланса классов. Если 80% автомобилей "хорошие", модель может предсказывать всех как "хорошие" и получить 80% accuracy, но не найти ни одного "проблемного" авто. |
| **Precision** | ❌ Важно найти все проблемные авто, даже ценой ложных срабатываний. Высокая Precision без Recall бесполезна. |
| **Recall** | ❌ Важно не пропустить проблемные авто, но слишком высокий Recall (ценой Precision) приведёт к множеству ложных срабатываний и потере денег на проверке хороших авто. |

### Почему F1 Score:

```
F1 = 2 * (Precision * Recall) / (Precision + Recall)
```

**F1 Score — это гармоническое среднее между Precision и Recall.**

1. **Баланс**: F1 учитывает и Precision, и Recall одновременно
2. **Штраф за дисбаланс**: Если одна метрика низкая, F1 тоже будет низким
3. **Бизнес-смысл**: 
   - **Recall** важен, чтобы не пропустить "лемон" (финансовые потери от продажи плохого авто)
   - **Precision** важен, чтобы не тратить ресурсы на проверку хороших авто
   - F1 находит золотую середину

### Дополнительные рекомендации:

```python
# Для бизнес-задачи можно использовать взвешенный F-beta:
# Fβ = (1 + β²) * (Precision * Recall) / (β² * Precision + Recall)
# 
# β > 1: больше фокус на Recall (найти все лемоны)
# β < 1: больше фокус на Precision (не ошибаться)
# β = 1: F1 Score (баланс)

# Для задачи lemon cars рекомендую β = 1.5
# Это означает, что Recall в 1.5 раза важнее Precision
```

### Итог:

| Метрика | Рекомендация |
|---------|-------------|
| **F1 Score** | ✅ для сбалансированного подхода |
| **F-beta (β=1.5)** | ✅ Если важно найти все проблемные авто |
| **AUPRC** | ✅ Для сравнения моделей (без порога) |